In [2]:
import polars as pl

auth = pl.scan_parquet("filtered_auth.parquet")

# Make sure timestamp is numeric or datetime
# If it is an int seconds column, keep it as int.
# If it is string, parse it.

In [21]:
import polars as pl

def norm_user(expr: pl.Expr) -> pl.Expr:
    return (
        expr.cast(pl.Utf8)
            .str.strip_chars()
            .str.split("@").list.first()
            .str.split("\\").list.last()
    )

feat_users = (
    train_df2.select(norm_user(pl.col("user")).alias("user"))
            .unique()
)

rt_users = (
    comp_users.select(norm_user(pl.col("src_user")).alias("user"))
             .unique()
)

overlap = feat_users.join(rt_users, on="user", how="inner").height
print("Overlapping users between features and redteam:", overlap)
print("Total feature users:", feat_users.height)
print("Total redteam users:", rt_users.height)

# Show a few examples from each side (sanity)
print("Sample feature users:", feat_users.head(10))
print("Sample redteam users:", rt_users.head(10))

Overlapping users between features and redteam: 98
Total feature users: 11376
Total redteam users: 98
Sample feature users: shape: (10, 1)
┌────────┐
│ user   │
│ ---    │
│ str    │
╞════════╡
│ U10726 │
│ U2097  │
│ U2736  │
│ U9224  │
│ U867   │
│ U4672  │
│ U3558  │
│ U5103  │
│ U3143  │
│ U5493  │
└────────┘
Sample redteam users: shape: (10, 1)
┌───────┐
│ user  │
│ ---   │
│ str   │
╞═══════╡
│ U1025 │
│ U882  │
│ U995  │
│ U3005 │
│ U3905 │
│ U212  │
│ U3277 │
│ U1048 │
│ U9407 │
│ U8777 │
└───────┘


In [3]:
auth.schema

C:\Users\kruti\AppData\Local\Temp\ipykernel_8824\736030822.py:1: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  auth.schema


Schema([('time', Int64),
        ('src_user', String),
        ('src_computer', String),
        ('dst_computer', String),
        ('logon_type', String),
        ('status', String)])

In [4]:
auth_df = auth.select(["time"]).collect(streaming=True)
q70 = auth_df["time"].quantile(0.70)
q85 = auth_df["time"].quantile(0.85)

train_auth = auth.filter(pl.col("time") <= q70)
val_auth   = auth.filter((pl.col("time") > q70) & (pl.col("time") <= q85))
test_auth  = auth.filter(pl.col("time") > q85)

C:\Users\kruti\AppData\Local\Temp\ipykernel_8824\2688365193.py:1: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  auth_df = auth.select(["time"]).collect(streaming=True)


In [5]:
train_users = train_auth.filter(pl.col("src_user").str.starts_with("U")).rename({"src_user": "user"})
val_users   = val_auth.filter(pl.col("src_user").str.starts_with("U")).rename({"src_user": "user"})
test_users  = test_auth.filter(pl.col("src_user").str.starts_with("U")).rename({"src_user": "user"})

In [6]:
dst_col = "dst_computer"   # change if needed

def make_user_features(df: pl.LazyFrame) -> pl.LazyFrame:
    return (
        df.group_by("user")
          .agg([
              pl.len().alias("total_logins"),
              (pl.col("status") == "Failure").sum().alias("failed_logins"),
              pl.col(dst_col).n_unique().alias("unique_destinations"),
              (pl.col("status") == "Failure").mean().alias("fail_rate"),
          ])
          .with_columns([
              (pl.col("failed_logins") / pl.col("total_logins")).fill_nan(0).alias("fail_rate_2")
          ])
    )

train_feat = make_user_features(train_users)
val_feat   = make_user_features(val_users)
test_feat  = make_user_features(test_users)

In [7]:
import polars as pl

red = pl.read_csv(
    "data_windows/redteam.txt.gz",
    has_header=False
)

red = red.rename({
    "column_1": "timestamp",
    "column_2": "src_user",
    "column_3": "src_computer",
    "column_4": "dst_computer",
})

red = red.with_columns(
    pl.col("timestamp").cast(pl.Int64)
)

red.write_parquet("redteam.parquet")

In [8]:
comp_users = (
    red.group_by("src_user")
       .agg(pl.len().alias("compromise_events"))
)

comp_users.write_parquet("comp_users.parquet")

In [9]:
import polars as pl

def attach_labels(feat: pl.LazyFrame) -> pl.DataFrame:
    out = (
        feat.join(
            comp_users.lazy(),
            left_on="user",
            right_on="src_user",
            how="left",
        )
        .with_columns(
            pl.col("compromise_events").fill_null(0)
        )
        .collect()
    )

    out = out.with_columns(
        (pl.col("compromise_events") > 0).cast(pl.Int8).alias("label")
    )
    return out

train_df = attach_labels(train_feat)
val_df   = attach_labels(val_feat)
test_df  = attach_labels(test_feat)

In [ ]:
print("train positives:", train_df["label"].sum())
print("val positives:", val_df["label"].sum())
print("test positives:", test_df["label"].sum())

In [10]:
def normalize_user(s: pl.Expr) -> pl.Expr:
    return s.str.strip_chars().str.split("@").list.first()

comp_users = comp_users.with_columns(normalize_user(pl.col("src_user")).alias("src_user"))

train_feat = train_feat.with_columns(normalize_user(pl.col("user")).alias("user"))
val_feat   = val_feat.with_columns(normalize_user(pl.col("user")).alias("user"))
test_feat  = test_feat.with_columns(normalize_user(pl.col("user")).alias("user"))

In [12]:
import numpy as np
from sklearn.preprocessing import RobustScaler

# Polars -> numpy
X_train = train_df.select(feature_cols).to_numpy()
X_val   = val_df.select(feature_cols).to_numpy()
X_test  = test_df.select(feature_cols).to_numpy()

y_val  = val_df["label"].to_numpy()
y_test = test_df["label"].to_numpy()

scaler = RobustScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

input_dim = X_train_s.shape[1]
print("input_dim:", input_dim)

NameError: name 'feature_cols' is not defined

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class NumpyDataset(Dataset):
    def __init__(self, X):
        self.X = torch.tensor(X, dtype=torch.float32)
    def __len__(self):
        return self.X.shape[0]
    def __getitem__(self, idx):
        return self.X[idx]

train_loader = DataLoader(NumpyDataset(X_train_s), batch_size=4096, shuffle=True, num_workers=0)
val_tensor   = torch.tensor(X_val_s, dtype=torch.float32)
test_tensor  = torch.tensor(X_test_s, dtype=torch.float32)

device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
import torch.nn as nn

class AutoEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, latent_dim=16, dropout=0.0):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, latent_dim),
            nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, input_dim),
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)

In [ ]:
import torch
import torch.optim as optim

def train_autoencoder(
    X_val_tensor,
    input_dim,
    hidden_dim,
    latent_dim,
    dropout,
    lr,
    weight_decay,
    epochs=50,
    patience=5,
):
    model = AutoEncoder(input_dim, hidden_dim=hidden_dim, latent_dim=latent_dim, dropout=dropout).to(device)
    opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()

    best_val = float("inf")
    best_state = None
    bad = 0

    for epoch in range(epochs):
        model.train()
        running = 0.0
        n = 0

        for xb in train_loader:
            xb = xb.to(device)
            opt.zero_grad(set_to_none=True)
            recon = model(xb)
            loss = loss_fn(recon, xb)
            loss.backward()
            opt.step()
            running += loss.item() * xb.size(0)
            n += xb.size(0)

        train_loss = running / max(n, 1)

        model.eval()
        with torch.no_grad():
            xv = X_val_tensor.to(device)
            rv = model(xv)
            val_loss = loss_fn(rv, xv).item()

        if val_loss < best_val - 1e-6:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, best_val

In [ ]:
def reconstruction_scores(model, X_tensor):
    model.eval()
    with torch.no_grad():
        x = X_tensor.to(device)
        r = model(x)
        err = ((r - x) ** 2).mean(dim=1)
        return err.detach().cpu().numpy()

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import roc_auc_score, average_precision_score
from torch.utils.data import Dataset, DataLoader

# ---- data -> numpy ----
X_train = train_df.select(feature_cols).to_numpy()
X_val   = val_df.select(feature_cols).to_numpy()
X_test  = test_df.select(feature_cols).to_numpy()

y_val  = val_df["label"].to_numpy()
y_test = test_df["label"].to_numpy()

# ---- scale ----
scaler = RobustScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

input_dim = X_train_s.shape[1]
print("input_dim:", input_dim)

# ---- torch ----
class NumpyDataset(Dataset):
    def __init__(self, X):
        self.X = torch.tensor(X, dtype=torch.float32)
    def __len__(self):
        return self.X.shape[0]
    def __getitem__(self, idx):
        return self.X[idx]

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

train_loader = DataLoader(NumpyDataset(X_train_s), batch_size=4096, shuffle=True, num_workers=0)
val_tensor   = torch.tensor(X_val_s, dtype=torch.float32)
test_tensor  = torch.tensor(X_test_s, dtype=torch.float32)

In [ ]:
class AutoEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, latent_dim=16, dropout=0.0):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, latent_dim),
            nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, input_dim),
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)

def reconstruction_scores(model, X_tensor):
    model.eval()
    with torch.no_grad():
        x = X_tensor.to(device)
        r = model(x)
        err = ((r - x) ** 2).mean(dim=1)
        return err.detach().cpu().numpy()

def precision_at_k(y_true, scores, k):
    k = min(k, len(scores))
    idx = np.argsort(scores)[::-1][:k]
    return float(y_true[idx].sum()) / float(k)

def train_autoencoder(
    train_loader,
    X_val_tensor,
    input_dim,
    hidden_dim,
    latent_dim,
    dropout,
    lr,
    weight_decay,
    epochs=50,
    patience=5,
):
    model = AutoEncoder(input_dim, hidden_dim=hidden_dim, latent_dim=latent_dim, dropout=dropout).to(device)
    opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()

    best_val = float("inf")
    best_state = None
    bad = 0

    for epoch in range(epochs):
        model.train()
        running = 0.0
        n = 0

        for xb in train_loader:
            xb = xb.to(device)
            opt.zero_grad(set_to_none=True)
            recon = model(xb)
            loss = loss_fn(recon, xb)
            loss.backward()
            opt.step()
            running += loss.item() * xb.size(0)
            n += xb.size(0)

        model.eval()
        with torch.no_grad():
            xv = X_val_tensor.to(device)
            rv = model(xv)
            val_loss = loss_fn(rv, xv).item()

        if val_loss < best_val - 1e-6:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, best_val

In [ ]:
search_space = {
    "hidden_dim": [32, 64, 128],
    "latent_dim": [4, 8, 16, 32],
    "dropout":    [0.0, 0.1, 0.2],
    "lr":         [1e-3, 3e-4, 1e-4],
    "weight_decay":[0.0, 1e-4, 1e-3],
    "epochs":     40,
    "patience":   5,
}

best_model = None
best_cfg = None
best_metrics = None

best_key = (-1.0, -1.0)  # (val_pr_auc, val_p@10)

trial = 0
total = (
    len(search_space["hidden_dim"]) *
    len(search_space["latent_dim"]) *
    len(search_space["dropout"]) *
    len(search_space["lr"]) *
    len(search_space["weight_decay"])
)

for hidden_dim in search_space["hidden_dim"]:
    for latent_dim in search_space["latent_dim"]:
        for dropout in search_space["dropout"]:
            for lr in search_space["lr"]:
                for wd in search_space["weight_decay"]:
                    trial += 1

                    model, best_val_recon = train_autoencoder(
                        train_loader=train_loader,
                        X_val_tensor=val_tensor,
                        input_dim=input_dim,
                        hidden_dim=hidden_dim,
                        latent_dim=latent_dim,
                        dropout=dropout,
                        lr=lr,
                        weight_decay=wd,
                        epochs=search_space["epochs"],
                        patience=search_space["patience"],
                    )

                    val_scores = reconstruction_scores(model, val_tensor)

                    # Handle edge case: if val has no positives or no negatives
                    if len(np.unique(y_val)) > 1:
                        val_roc = roc_auc_score(y_val, val_scores)
                        val_pr  = average_precision_score(y_val, val_scores)
                    else:
                        val_roc = np.nan
                        val_pr  = np.nan

                    p10 = precision_at_k(y_val, val_scores, 10)
                    p50 = precision_at_k(y_val, val_scores, 50)

                    key = (val_pr, p10)

                    if (val_pr > best_key[0]) or (np.isclose(val_pr, best_key[0]) and p10 > best_key[1]):
                        best_key = key
                        best_model = model  # keep only this model
                        best_cfg = {
                            "hidden_dim": hidden_dim,
                            "latent_dim": latent_dim,
                            "dropout": dropout,
                            "lr": lr,
                            "weight_decay": wd,
                        }
                        best_metrics = {
                            "best_val_recon": best_val_recon,
                            "val_roc_auc": val_roc,
                            "val_pr_auc": val_pr,
                            "val_p@10": p10,
                            "val_p@50": p50,
                        }

                        print(f"[{trial}/{total}] NEW BEST")
                        print("cfg:", best_cfg)
                        print("metrics:", best_metrics)
                    else:
                        # lightweight progress print every so often
                        if trial % 10 == 0:
                            print(f"[{trial}/{total}] current best val_pr={best_key[0]:.6f}, val_p@10={best_key[1]:.4f}")

In [ ]:
print("FINAL BEST CONFIG:", best_cfg)
print("FINAL BEST VAL METRICS:", best_metrics)

In [ ]:
test_scores = reconstruction_scores(best_model, test_tensor)

test_roc = roc_auc_score(y_test, test_scores)
test_pr  = average_precision_score(y_test, test_scores)

print("TEST ROC-AUC:", test_roc)
print("TEST PR-AUC:", test_pr)

for k in [10, 20, 50, 100, 200]:
    print(f"TEST Precision@{k}: {precision_at_k(y_test, test_scores, k):.4f}")

BETTER USER FEATURES

In [13]:
import polars as pl

def user_features(ldf: pl.LazyFrame) -> pl.LazyFrame:
    # filter to real users
    ldf = ldf.filter(pl.col("src_user").str.starts_with("U")).with_columns(
        pl.col("src_user").alias("user")
    )

    # add hour + weekend flags (LANL time is seconds, so hour = (time/3600) % 24)
    ldf = ldf.with_columns([
        ((pl.col("time") // 3600) % 24).cast(pl.Int32).alias("hour"),
        ((pl.col("time") // 86400) % 7).cast(pl.Int32).alias("dow"),
    ]).with_columns([
        (pl.col("dow") >= 5).cast(pl.Int8).alias("is_weekend"),
        (pl.col("hour").is_between(0, 5)).cast(pl.Int8).alias("is_night"),
        (pl.col("status") == "Failure").cast(pl.Int8).alias("is_fail"),
    ])

    # per-user time gap stats (burstiness)
    gaps = (
        ldf.select(["user", "time"])
           .sort(["user", "time"])
           .with_columns([
               (pl.col("time") - pl.col("time").shift(1)).over("user").alias("dt")
           ])
           .group_by("user")
           .agg([
               pl.col("dt").drop_nulls().median().alias("median_dt"),
               pl.col("dt").drop_nulls().mean().alias("mean_dt"),
               pl.col("dt").drop_nulls().std().alias("std_dt"),
               pl.col("dt").drop_nulls().quantile(0.90).alias("p90_dt"),
           ])
    )

    # destination concentration: how "peaky" destinations are
    dest_share = (
        ldf.group_by(["user", "dst_computer"])
           .agg(pl.len().alias("cnt"))
           .group_by("user")
           .agg([
               pl.col("cnt").max().alias("top_dst_cnt"),
               pl.col("cnt").sum().alias("dst_total_cnt"),
               pl.col("cnt").mean().alias("dst_mean_cnt"),
               pl.col("cnt").std().alias("dst_std_cnt"),
           ])
           .with_columns([
               (pl.col("top_dst_cnt") / (pl.col("dst_total_cnt") + 1e-6)).alias("top_dst_share")
           ])
    )

    # logon_type distribution (counts + diversity)
    logon_div = (
        ldf.group_by(["user", "logon_type"])
           .agg(pl.len().alias("lt_cnt"))
           .group_by("user")
           .agg([
               pl.col("logon_type").n_unique().alias("unique_logon_types"),
               pl.col("lt_cnt").max().alias("top_logon_cnt"),
               pl.col("lt_cnt").sum().alias("logon_total_cnt"),
           ])
           .with_columns([
               (pl.col("top_logon_cnt") / (pl.col("logon_total_cnt") + 1e-6)).alias("top_logon_share")
           ])
    )

    # core aggregates
    core = (
        ldf.group_by("user")
           .agg([
               pl.len().alias("total_logins"),
               pl.col("is_fail").sum().alias("failed_logins"),
               pl.col("dst_computer").n_unique().alias("unique_destinations"),
               pl.col("src_computer").n_unique().alias("unique_src_computers"),
               pl.col("is_fail").mean().alias("fail_rate"),
               pl.col("is_night").mean().alias("night_rate"),
               pl.col("is_weekend").mean().alias("weekend_rate"),
               pl.col("hour").n_unique().alias("unique_hours"),
           ])
           .with_columns([
               (pl.col("failed_logins") / (pl.col("total_logins") + 1e-6)).alias("fail_rate_2"),
               (pl.col("unique_destinations") / (pl.col("total_logins") + 1e-6)).alias("dst_per_login"),
               (pl.col("unique_src_computers") / (pl.col("total_logins") + 1e-6)).alias("src_per_login"),
           ])
    )

    # join everything
    feat = (
        core.join(gaps, on="user", how="left")
            .join(dest_share, on="user", how="left")
            .join(logon_div, on="user", how="left")
            .with_columns([
                pl.all().exclude("user").fill_null(0)
            ])
    )

    return feat

In [14]:
train_feat = user_features(train_auth)
val_feat   = user_features(val_auth)
test_feat  = user_features(test_auth)

In [15]:
import polars as pl

def attach_labels(feat: pl.LazyFrame) -> pl.DataFrame:
    out = (
        feat.join(comp_users.lazy(), left_on="user", right_on="src_user", how="left")
            .with_columns(pl.col("compromise_events").fill_null(0))
            .collect()
    )
    out = out.with_columns((pl.col("compromise_events") > 0).cast(pl.Int8).alias("label"))
    return out

train_df = attach_labels(train_feat)
val_df   = attach_labels(val_feat)
test_df  = attach_labels(test_feat)

In [16]:
# Pick initial feature columns (everything except user/labels)
base_feature_cols = [c for c in train_df.columns if c not in ("user", "label", "compromise_events", "src_user")]

baseline = (
    train_df.select(["user"] + base_feature_cols)
            .group_by("user")
            .agg([pl.col(c).mean().alias(f"base_{c}") for c in base_feature_cols])
)

def add_deviation(cur_df: pl.DataFrame) -> pl.DataFrame:
    out = cur_df.join(baseline, on="user", how="left")
    for c in base_feature_cols:
        out = out.with_columns([
            (pl.col(c) - pl.col(f"base_{c}")).fill_null(0).alias(f"diff_{c}"),
            (pl.col(c) / (pl.col(f"base_{c}") + 1e-6)).fill_null(0).alias(f"ratio_{c}"),
        ])
    # keep it tidy
    return out.drop([f"base_{c}" for c in base_feature_cols])

train_df2 = add_deviation(train_df)
val_df2   = add_deviation(val_df)
test_df2  = add_deviation(test_df)

feature_cols2 = [c for c in train_df2.columns if c not in ("user", "label", "compromise_events", "src_user")]
print("num features:", len(feature_cols2))

num features: 72


In [17]:
import numpy as np
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score, average_precision_score

def precision_at_k(y_true, scores, k):
    k = min(k, len(scores))
    idx = np.argsort(scores)[::-1][:k]
    return float(y_true[idx].sum()) / float(k)

X_train = train_df2.select(feature_cols2).to_numpy()
X_val   = val_df2.select(feature_cols2).to_numpy()
X_test  = test_df2.select(feature_cols2).to_numpy()

y_val  = val_df2["label"].to_numpy()
y_test = test_df2["label"].to_numpy()

scaler = RobustScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

iso = IsolationForest(n_estimators=500, contamination=0.01, random_state=42, n_jobs=-1)
iso.fit(X_train_s)

val_scores  = -iso.decision_function(X_val_s)
test_scores = -iso.decision_function(X_test_s)

print("VAL ROC-AUC:", roc_auc_score(y_val, val_scores))
print("VAL PR-AUC:", average_precision_score(y_val, val_scores))
print("TEST ROC-AUC:", roc_auc_score(y_test, test_scores))
print("TEST PR-AUC:", average_precision_score(y_test, test_scores))

for k in [10, 20, 50, 100, 200]:
    print(f"TEST Precision@{k}: {precision_at_k(y_test, test_scores, k):.4f}")

VAL ROC-AUC: nan
VAL PR-AUC: 0.0
TEST ROC-AUC: nan
TEST PR-AUC: 0.0
TEST Precision@10: 0.0000
TEST Precision@20: 0.0000
TEST Precision@50: 0.0000
TEST Precision@100: 0.0000
TEST Precision@200: 0.0000


e:\Projects\datascience\venv\Lib\site-packages\sklearn\metrics\_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
e:\Projects\datascience\venv\Lib\site-packages\sklearn\metrics\_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
e:\Projects\datascience\venv\Lib\site-packages\sklearn\metrics\_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
e:\Projects\datascience\venv\Lib\site-packages\sklearn\metrics\_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


In [18]:
# Autoencoder re-run on the NEW feature set (train_df2/val_df2/test_df2 + feature_cols2)
# - stores ONLY the best model so far
# - uses MAE reconstruction scoring (more robust)
# - searches a tight space that usually works better than your previous AE config

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import roc_auc_score, average_precision_score
from torch.utils.data import Dataset, DataLoader

# ----------------------------
# 1) Prepare arrays + scaling
# ----------------------------
X_train = train_df2.select(feature_cols2).to_numpy()
X_val   = val_df2.select(feature_cols2).to_numpy()
X_test  = test_df2.select(feature_cols2).to_numpy()

y_val  = val_df2["label"].to_numpy()
y_test = test_df2["label"].to_numpy()

scaler = RobustScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

input_dim = X_train_s.shape[1]
print("input_dim:", input_dim)

class NumpyDataset(Dataset):
    def __init__(self, X):
        self.X = torch.tensor(X, dtype=torch.float32)
    def __len__(self):
        return self.X.shape[0]
    def __getitem__(self, idx):
        return self.X[idx]

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

train_loader = DataLoader(NumpyDataset(X_train_s), batch_size=4096, shuffle=True, num_workers=0)
val_tensor   = torch.tensor(X_val_s, dtype=torch.float32)
test_tensor  = torch.tensor(X_test_s, dtype=torch.float32)

# ----------------------------
# 2) Model
# ----------------------------
class AutoEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, latent_dim=16, dropout=0.1):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, latent_dim),
            nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, input_dim),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

# ----------------------------
# 3) Helpers
# ----------------------------
def precision_at_k(y_true, scores, k):
    k = min(k, len(scores))
    idx = np.argsort(scores)[::-1][:k]
    return float(y_true[idx].sum()) / float(k)

def reconstruction_scores_mae(model, X_tensor):
    model.eval()
    with torch.no_grad():
        x = X_tensor.to(device)
        r = model(x)
        err = (r - x).abs().mean(dim=1)  # MAE per row
        return err.detach().cpu().numpy()

def train_autoencoder(
    train_loader,
    X_val_tensor,
    input_dim,
    hidden_dim,
    latent_dim,
    dropout,
    lr,
    weight_decay,
    epochs=50,
    patience=5,
):
    model = AutoEncoder(input_dim, hidden_dim=hidden_dim, latent_dim=latent_dim, dropout=dropout).to(device)
    opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    # Train with MSE for stability, score with MAE for robustness
    loss_fn = nn.MSELoss()

    best_val = float("inf")
    best_state = None
    bad = 0

    for epoch in range(epochs):
        model.train()
        for xb in train_loader:
            xb = xb.to(device)
            opt.zero_grad(set_to_none=True)
            recon = model(xb)
            loss = loss_fn(recon, xb)
            loss.backward()
            opt.step()

        model.eval()
        with torch.no_grad():
            xv = X_val_tensor.to(device)
            rv = model(xv)
            val_loss = loss_fn(rv, xv).item()

        if val_loss < best_val - 1e-6:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, best_val

# ----------------------------
# 4) Tight sweep: store only best model so far
# ----------------------------
search_space = {
    "hidden_dim": [64, 128],
    "latent_dim": [8, 16],      # constrained latent (important)
    "dropout":    [0.1, 0.2],
    "lr":         [1e-3, 3e-4],
    "weight_decay":[0.0, 1e-4],
    "epochs":     40,
    "patience":   5,
}

best_model = None
best_cfg = None
best_metrics = None
best_key = (-1.0, -1.0)  # (val_pr_auc, val_p@10)

trial = 0
total = (
    len(search_space["hidden_dim"]) *
    len(search_space["latent_dim"]) *
    len(search_space["dropout"]) *
    len(search_space["lr"]) *
    len(search_space["weight_decay"])
)

for hidden_dim in search_space["hidden_dim"]:
    for latent_dim in search_space["latent_dim"]:
        for dropout in search_space["dropout"]:
            for lr in search_space["lr"]:
                for wd in search_space["weight_decay"]:
                    trial += 1

                    model, best_val_recon = train_autoencoder(
                        train_loader=train_loader,
                        X_val_tensor=val_tensor,
                        input_dim=input_dim,
                        hidden_dim=hidden_dim,
                        latent_dim=latent_dim,
                        dropout=dropout,
                        lr=lr,
                        weight_decay=wd,
                        epochs=search_space["epochs"],
                        patience=search_space["patience"],
                    )

                    val_scores = reconstruction_scores_mae(model, val_tensor)

                    if len(np.unique(y_val)) > 1:
                        val_roc = roc_auc_score(y_val, val_scores)
                        val_pr  = average_precision_score(y_val, val_scores)
                    else:
                        val_roc = np.nan
                        val_pr  = np.nan

                    p10 = precision_at_k(y_val, val_scores, 10)
                    p50 = precision_at_k(y_val, val_scores, 50)

                    key = (val_pr, p10)

                    if (val_pr > best_key[0]) or (np.isclose(val_pr, best_key[0]) and p10 > best_key[1]):
                        best_key = key
                        best_model = model
                        best_cfg = {
                            "hidden_dim": hidden_dim,
                            "latent_dim": latent_dim,
                            "dropout": dropout,
                            "lr": lr,
                            "weight_decay": wd,
                        }
                        best_metrics = {
                            "best_val_recon_mse": best_val_recon,
                            "val_roc_auc": val_roc,
                            "val_pr_auc": val_pr,
                            "val_p@10": p10,
                            "val_p@50": p50,
                        }

                        print(f"[{trial}/{total}] NEW BEST")
                        print("cfg:", best_cfg)
                        print("metrics:", best_metrics)
                    else:
                        if trial % 5 == 0:
                            print(f"[{trial}/{total}] current best val_pr={best_key[0]:.6f}, val_p@10={best_key[1]:.4f}")

print("\nFINAL BEST CONFIG:", best_cfg)
print("FINAL BEST VAL METRICS:", best_metrics)

# ----------------------------
# 5) Final evaluation on TEST
# ----------------------------
test_scores = reconstruction_scores_mae(best_model, test_tensor)

test_roc = roc_auc_score(y_test, test_scores)
test_pr  = average_precision_score(y_test, test_scores)

print("\nTEST ROC-AUC:", test_roc)
print("TEST PR-AUC:", test_pr)

for k in [10, 20, 50, 100, 200]:
    print(f"TEST Precision@{k}: {precision_at_k(y_test, test_scores, k):.4f}")

input_dim: 72
device: cpu
[5/32] current best val_pr=-1.000000, val_p@10=-1.0000
[10/32] current best val_pr=-1.000000, val_p@10=-1.0000
[15/32] current best val_pr=-1.000000, val_p@10=-1.0000
[20/32] current best val_pr=-1.000000, val_p@10=-1.0000
[25/32] current best val_pr=-1.000000, val_p@10=-1.0000
[30/32] current best val_pr=-1.000000, val_p@10=-1.0000

FINAL BEST CONFIG: None
FINAL BEST VAL METRICS: None


AttributeError: 'NoneType' object has no attribute 'eval'

In [19]:
import numpy as np

print("VAL label uniques:", np.unique(y_val, return_counts=True))
print("TEST label uniques:", np.unique(y_test, return_counts=True))

VAL label uniques: (array([0], dtype=int8), array([15880]))
TEST label uniques: (array([0], dtype=int8), array([17607]))


In [20]:
import polars as pl

feat_users = train_df2.select("user").unique()
rt_users = comp_users.select(pl.col("src_user").alias("user")).unique()

overlap = feat_users.join(rt_users, on="user", how="inner").height
print("Overlapping users between features and redteam:", overlap)
print("Total feature users:", feat_users.height)
print("Total redteam users:", rt_users.height)

Overlapping users between features and redteam: 0
Total feature users: 23803
Total redteam users: 98
